# Python 装饰器完全指南

> 目标：从「语法糖本质」到「工程实践」，逐个知识点配合**可运行示例**击破。
> 阅读方式：按顺序执行每个代码单元，观察输出，再对照上面的原理说明。

---

## 一句话本质

> **装饰器是一个可调用对象：接收一个函数（或类），返回一个新的函数（或类）。**

```python
@decorator
def func(): ...

# 完全等价于
def func(): ...
func = decorator(func)
```

---

## 0. 前置知识：三个必须掌握的语言特性

装饰器不是新语法，而是三个既有特性的组合：

1. **函数是一等对象（first-class object）**
   函数可以被赋值给变量、作为参数传递、作为返回值、存进容器。
2. **闭包（closure）**
   内层函数可以记住并访问外层函数的局部变量，即使外层函数已经返回。
3. **高阶函数（higher-order function）**
   接收函数作为参数，或返回函数的函数。

装饰器 = 高阶函数 + 闭包 + 一等函数。下面逐一验证。

In [5]:
# ---------- 特性 1：函数是一等对象 ----------
def greet(name):
    """返回一句问候语。"""
    return f"Hello, {name}"


# 1) 把函数赋值给变量
say = greet
print("赋值给变量:", say("Alice"))

# 2) 把函数作为参数传递
def call_twice(fn, arg):
    return [fn(arg), fn(arg)]

print("作为参数:", call_twice(say, "Bob"))

# 3) 把函数放进容器
registry = {"greet": greet}
print("存入字典:", registry["greet"]("Carol"))

# 4) 函数自身也是对象，拥有属性
print("函数对象属性 __name__:", greet.__name__, "| __doc__:", greet.__doc__)

赋值给变量: Hello, Alice
作为参数: ['Hello, Bob', 'Hello, Bob']
存入字典: Hello, Carol
函数对象属性 __name__: greet | __doc__: 返回一句问候语。


In [8]:
# ---------- 特性 2：闭包 ----------
def make_multiplier(factor):
    """外层函数：负责接收配置。"""
    def mul(x):
        # factor 来自外层作用域，mul 返回后依然记得它 —— 这就是闭包
        return x * factor
    return mul


times3 = make_multiplier(3)
times10 = make_multiplier(10)

print("times3(7) =", times3(7))
print("times10(7) =", times10(7))

# 闭包捕获的外部变量存放在 __closure__ 中
print("times3 捕获的外部变量:", times3.__closure__[0].cell_contents)
print("times10 捕获的外部变量:", times10.__closure__[0].cell_contents)

times3(7) = 21
times10(7) = 70
times3 捕获的外部变量: 3
times10 捕获的外部变量: 10


In [3]:
# ---------- 特性 3：高阶函数 ----------
def apply(func, seq):
    """接收一个函数，把它应用到序列的每个元素上。"""
    return [func(x) for x in seq]


print(apply(abs, [-1, 2, -3]))
print(apply(str.upper, ["llm", "ops"]))

# 返回函数的高阶函数，就是"装饰器的雏形"
def add_prefix(prefix):
    def decorator(func):
        def wrapper(*args, **kwargs):
            return prefix + func(*args, **kwargs)
        return wrapper
    return decorator


@add_prefix("[INFO] ")
def build(msg):
    return msg

print(build("模型加载完成"))

[1, 2, 3]
['LLM', 'OPS']
[INFO] 模型加载完成


---

## 1. `@` 语法糖的本质

`@decorator` 没有任何魔法，它只是把「赋值」这一步隐藏了：

```python
@decorator
def func(): ...
```

等价于：

```python
def func(): ...
func = decorator(func)     # 名字 func 被重新绑定到了返回值
```

**两个必须记住的结论：**

1. **装饰器在「函数定义时」就执行了**（模块 import 阶段），不是调用时。
2. `func` 这个名字之后指向的是 `decorator(func)` 的返回值（通常是 wrapper），
   原函数只能通过闭包访问。

In [4]:
def shout(func):
    """最简单的装饰器：调用前后各打印一行。"""
    def wrapper(*args, **kwargs):
        print(">>> before")
        result = func(*args, **kwargs)
        print(">>> after")
        return result
    return wrapper


def hello(name):
    print(f"Hello, {name}")


# 写法 A：手动包装（不用语法糖）
manual_wrapped = shout(hello)
print("--- 手动包装 ---")
manual_wrapped("Alice")
print("包装后对象:", manual_wrapped, "| __name__ =", manual_wrapped.__name__)


# 写法 B：语法糖
@shout
def hello_decorated(name):
    print(f"Hello, {name}")


print("\n--- 语法糖 ---")
hello_decorated("Bob")
print("装饰后 __name__ =", hello_decorated.__name__)

# 验证两者等价：手动包装后的行为与语法糖完全一致
print("\n两者等价:", manual_wrapped.__name__ == hello_decorated.__name__)

--- 手动包装 ---
>>> before
Hello, Alice
>>> after
包装后对象: <function shout.<locals>.wrapper at 0x1077bc040> | __name__ = wrapper

--- 语法糖 ---
>>> before
Hello, Bob
>>> after
装饰后 __name__ = wrapper

两者等价: True


In [5]:
# 原函数并没有消失，它被闭包保存了下来
print("wrapper 闭包中保存的原函数:", hello_decorated.__closure__[0].cell_contents)
print("直接调用原函数（绕过装饰）:", hello_decorated.__closure__[0].cell_contents("Carol"))

# 更优雅的做法：用 __wrapped__（配合 functools.wraps，见第 3 节）

wrapper 闭包中保存的原函数: <function hello_decorated at 0x1077bc220>
Hello, Carol
直接调用原函数（绕过装饰）: None


---

## 2. 第一个函数装饰器

标准骨架只有 5 行，**三个要点缺一不可**：

```python
def decorator(func):          # 1) 接收被装饰函数
    def wrapper(*args, **kwargs):   # 2) 用 *args/**kwargs 接收任意参数
        ...
        result = func(*args, **kwargs)
        ...
        return result         # 3) 必须 return，否则调用方拿到 None
    return wrapper            # 4) 返回 wrapper
```

其中 `*args, **kwargs` 是「透传」的关键——装饰器和被装饰函数的签名**解耦**，
同一个装饰器才能用在任意函数上。

In [6]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("调用前")
        result = func(*args, **kwargs)
        print("调用后")
        return result
    return wrapper


@my_decorator
def say_hello(name):
    print(f"Hello, {name}")
    return name


ret = say_hello("Alice")
print("返回值:", ret)

调用前
Hello, Alice
调用后
返回值: Alice


In [7]:
# ---------- 反例：忘记 return result ----------
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)      # 调用了，但没有返回
    return wrapper


@bad_decorator
def add(a, b):
    return a + b


print("bad_decorator 的返回值:", add(1, 2))       # None
print("正确装饰器的返回值:", my_decorator(lambda a, b: a + b)(1, 2))

bad_decorator 的返回值: None
调用前
调用后
正确装饰器的返回值: 3


---

## 3. `functools.wraps`：保住函数的「身份证」

不加 `wraps` 会丢失：

| 属性 | 影响 |
| --- | --- |
| `__name__` / `__qualname__` | 日志、报错信息里全是 `wrapper` |
| `__doc__` | `help()`、文档生成工具失效 |
| `__module__` | 定位不到源码位置 |
| `__wrapped__` | `inspect.signature()` 拿到的是 `(*args, **kwargs)` |
| 类型注解 | IDE / 类型检查器无法推断 |

`@wraps(func)` 本质是把 `func` 的这些属性**复制**到 `wrapper` 上，
并设置 `wrapper.__wrapped__ = func`。

In [8]:
from functools import wraps
import inspect


# ---------- 不加 wraps ----------
def no_wraps_deco(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


# ---------- 加 wraps ----------
def with_wraps_deco(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


def sample(name: str, age: int = 18) -> str:
    """示例函数：返回自我介绍。"""
    return f"{name} is {age}"


bad = no_wraps_deco(sample)
good = with_wraps_deco(sample)

print("=== 不加 wraps ===")
print("__name__:", bad.__name__)
print("__doc__:", bad.__doc__)
print("signature:", inspect.signature(bad))

print("\n=== 加 wraps ===")
print("__name__:", good.__name__)
print("__doc__:", good.__doc__)
print("signature:", inspect.signature(good))
print("__wrapped__ 指向原函数:", good.__wrapped__ is sample)
print("注解:", good.__annotations__)

=== 不加 wraps ===
__name__: wrapper
__doc__: None
signature: (*args, **kwargs)

=== 加 wraps ===
__name__: sample
__doc__: 示例函数：返回自我介绍。
signature: (name: str, age: int = 18) -> str
__wrapped__ 指向原函数: True
注解: {'name': <class 'str'>, 'age': <class 'int'>, 'return': <class 'str'>}


In [9]:
# 实用装饰器：日志 + wraps 的标准写法
def log_calls(func):
    """打印函数调用的入参与返回值。"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] 调用 {func.__name__}(args={args}, kwargs={kwargs})")
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__} 返回 {result!r}")
        return result
    return wrapper


@log_calls
def divide(a, b):
    """除法。"""
    return a / b


print(divide(10, 4))
print("元信息仍然保留:", divide.__name__, "|", divide.__doc__)

[LOG] 调用 divide(args=(10, 4), kwargs={})
[LOG] divide 返回 2.5
2.5
元信息仍然保留: divide | 除法。


---

## 4. 带参数的装饰器（装饰器工厂）

对比两种调用形式：

```python
@my_decorator          # 不带参：decorator(func)
def f(): ...

@repeat(3)             # 带参：先 repeat(3) 得到真正的 decorator，再 decorator(f)
def g(): ...
```

所以带参装饰器要**多一层嵌套**：

```python
def 装饰器工厂(参数...):      # 第 1 层：接收装饰器自己的参数
    def decorator(func):      # 第 2 层：接收被装饰函数
        @wraps(func)
        def wrapper(*args, **kwargs):   # 第 3 层：真正被调用的函数
            ...
            return func(*args, **kwargs)
        return wrapper
    return decorator
```

等价展开：`greet = repeat(3)(greet)`。

> 记忆口诀：**要传参，多一层。**

In [10]:
def repeat(times):
    """装饰器工厂：让被装饰函数重复执行 times 次，返回最后一次结果。"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator


@repeat(3)
def greet(name):
    print(f"Hi, {name}")
    return len(name)


print("返回值（最后一次）:", greet("Bob"))

Hi, Bob
Hi, Bob
Hi, Bob
返回值（最后一次）: 3


In [11]:
# 等价写法验证
def _greet(name):
    print(f"Hi, {name}")
    return len(name)


manual = repeat(3)(_greet)      # 工厂调用 + 装饰器调用
print("=== repeat(3)(_greet) ===")
print(manual("Alice"))

# 观察执行链：repeat(3) 先返回 decorator
factory_result = repeat(3)
print("\nrepeat(3) 返回的是:", factory_result)
print("factory_result(_greet) 返回的是:", factory_result(_greet))
print("注意：因为用了 @wraps，wrapper 的 __name__ 已被改成 _greet：", factory_result(_greet).__name__)

=== repeat(3)(_greet) ===
Hi, Alice
Hi, Alice
Hi, Alice
5

repeat(3) 返回的是: <function repeat.<locals>.decorator at 0x1077bd300>
factory_result(_greet) 返回的是: <function _greet at 0x1077bd260>
注意：因为用了 @wraps，wrapper 的 __name__ 已被改成 _greet： _greet


In [12]:
# 进阶：带可选参数的装饰器（支持 @repeat 和 @repeat(3) 两种写法）
def flexible_repeat(_func=None, *, times=2):
    """同时支持 @flexible_repeat 与 @flexible_repeat(times=5)。"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper

    if _func is None:
        return decorator          # 用法：@flexible_repeat(times=5)
    return decorator(_func)       # 用法：@flexible_repeat


@flexible_repeat
def a():
    print("a", end=" ")


@flexible_repeat(times=4)
def b():
    print("b", end=" ")


print("默认次数 2:"); a()
print("\n指定次数 4:"); b()

默认次数 2:
a a 
指定次数 4:
b b b b 

---

## 5. 多个装饰器的执行顺序

规则只有一句：

```python
@deco1
@deco2
def f(): ...

# 等价于
f = deco1(deco2(f))
```

- **包装阶段（定义时）**：离函数**近**的先执行 → `deco2` 先包装 `f`，再被 `deco1` 包装。
- **调用阶段（运行时）**：离函数**远**的先执行 → `deco1` 的 before 先打印，after 后打印。

可以理解为「洋葱模型」：外面那层（`deco1`）最先接触调用方。

In [13]:
def deco1(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("deco1 before")
        result = func(*args, **kwargs)
        print("deco1 after")
        return result
    return wrapper


def deco2(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("deco2 before")
        result = func(*args, **kwargs)
        print("deco2 after")
        return result
    return wrapper


@deco1
@deco2
def f():
    print("  [被装饰的函数 f]")


print("=== 调用 f() ===")
f()

print("\n=== 展开验证 ===")
def raw():
    print("  [被装饰的函数 f]")

manual = deco1(deco2(raw))
manual()

=== 调用 f() ===
deco1 before
deco2 before
  [被装饰的函数 f]
deco2 after
deco1 after

=== 展开验证 ===
deco1 before
deco2 before
  [被装饰的函数 f]
deco2 after
deco1 after


In [14]:
# 洋葱模型可视化：三层装饰器
def layer(name):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(f"  {name} 进入")
            result = func(*args, **kwargs)
            print(f"  {name} 退出")
            return result
        return wrapper
    return decorator


@layer("第1层(最外)")
@layer("第2层")
@layer("第3层(最内)")
def core():
    print("    >>> 核心逻辑 <<<")


core()

  第1层(最外) 进入
  第2层 进入
  第3层(最内) 进入
    >>> 核心逻辑 <<<
  第3层(最内) 退出
  第2层 退出
  第1层(最外) 退出


---

## 6. 用类实现装饰器

装饰器只要求是「可调用对象」。实现了 `__call__` 的类实例同样可调用，
因此类也能当装饰器。

```python
class Decorator:
    def __init__(self, func):        # 装饰时执行：func 被存为实例属性
        self.func = func
    def __call__(self, *args, **kwargs):   # 每次调用时执行
        return self.func(*args, **kwargs)
```

**什么时候用类而不用函数？**

- 需要**保存状态**（调用次数、缓存、最近一次耗时…）→ 用实例属性，比闭包变量更直观、可外部访问。
- 需要**额外方法**（如 `reset()`、`stats()`）暴露给调用方。
- 带参数的类装饰器更清爽：`__init__` 收参数，`__call__` 收函数。

类装饰器别忘了用 `functools.update_wrapper(self, func)` 复制元信息。

In [15]:
from functools import update_wrapper


class CountCalls:
    """统计函数被调用的次数。"""
    def __init__(self, func):
        self.func = func
        self.count = 0
        update_wrapper(self, func)      # 类的等价物：复制 __name__/__doc__ 等

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"{self.func.__name__} 被调用 {self.count} 次")
        return self.func(*args, **kwargs)

    def reset(self):
        """类装饰器独有的优势：可以暴露额外方法。"""
        self.count = 0


@CountCalls
def say(name):
    print(f"Hello, {name}")


say("Alice")
say("Bob")
say("Carol")
print("当前计数:", say.count)
print("元信息:", say.__name__)

say.reset()
print("reset 后计数:", say.count)

say 被调用 1 次
Hello, Alice
say 被调用 2 次
Hello, Bob
say 被调用 3 次
Hello, Carol
当前计数: 3
元信息: say
reset 后计数: 0


In [16]:
# 带参数的类装饰器：__init__ 收参数，__call__ 收函数
class Retry:
    """失败自动重试 times 次。"""
    def __init__(self, times=3, exceptions=(Exception,)):
        self.times = times
        self.exceptions = exceptions

    def __call__(self, func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(1, self.times + 1):
                try:
                    return func(*args, **kwargs)
                except self.exceptions as e:
                    if i == self.times:
                        raise
                    print(f"  第 {i} 次失败: {e}，重试中...")
        return wrapper


attempt = 0

@Retry(times=3, exceptions=(ValueError,))
def flaky():
    global attempt
    attempt += 1
    if attempt < 3:
        raise ValueError(f"模拟失败 #{attempt}")
    return "成功"


print("结果:", flaky())
print("实际尝试次数:", attempt)

  第 1 次失败: 模拟失败 #1，重试中...
  第 2 次失败: 模拟失败 #2，重试中...
结果: 成功
实际尝试次数: 3


---

## 7. 装饰类（Class Decorator）

装饰器的返回值也可以是**类**。它在类定义完成后立即执行，常用于：

1. 给类**动态注入方法 / 属性**
2. 把类**注册**到某个注册表（插件系统的基础）
3. **修改**已有方法（如统一加日志）
4. 像 `@dataclass` 一样**自动生成**样板代码

In [17]:
# 用法 1：动态添加方法
def add_str_method(cls):
    def __str__(self):
        return f"{self.__class__.__name__}({self.__dict__})"
    cls.__str__ = __str__
    return cls


@add_str_method
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y


p = Point(1, 2)
print(p)
print(p.__dict__)

Point({'x': 1, 'y': 2})
{'x': 1, 'y': 2}


In [18]:
# 用法 2：类注册器（插件系统的核心模式）
MODEL_REGISTRY = {}


def register_model(name):
    """把类注册进 MODEL_REGISTRY，并返回原类（不修改行为）。"""
    def decorator(cls):
        if name in MODEL_REGISTRY:
            raise ValueError(f"模型 {name} 已注册")
        MODEL_REGISTRY[name] = cls
        return cls            # 注意：注册型装饰器通常原样返回 cls
    return decorator


@register_model("gpt-4o")
class GPT4o:
    def invoke(self, prompt):
        return f"[gpt-4o] {prompt}"


@register_model("qwen-max")
class QwenMax:
    def invoke(self, prompt):
        return f"[qwen-max] {prompt}"


print("注册表:", MODEL_REGISTRY)
# 运行时按名字动态取用
model_cls = MODEL_REGISTRY["qwen-max"]
print(model_cls().invoke("你好"))

注册表: {'gpt-4o': <class '__main__.GPT4o'>, 'qwen-max': <class '__main__.QwenMax'>}
[qwen-max] 你好


In [19]:
# 用法 3：统一给类的所有公开方法加日志
def log_all_methods(cls):
    for attr_name, attr in vars(cls).items():
        if callable(attr) and not attr_name.startswith("__"):
            setattr(cls, attr_name, log_calls(attr))   # 复用第 3 节的 log_calls
    return cls


@log_all_methods
class UserService:
    def create(self, name):
        return f"created {name}"

    def delete(self, name):
        return f"deleted {name}"


svc = UserService()
svc.create("Alice")

[LOG] 调用 create(args=(<__main__.UserService object at 0x1077a52b0>, 'Alice'), kwargs={})
[LOG] create 返回 'created Alice'


'created Alice'

---

## 8. 实用示例集

下面 5 个是工程里真正高频的装饰器，都可以直接复制到项目中使用。

### 8.1 计时器 `timer`

要点：用 `try/finally` 保证**即使函数抛异常也能打印耗时**；
计时用 `time.perf_counter()`（单调、高精度），不要用 `time.time()`。

In [20]:
import time


def timer(func):
    """打印函数执行耗时。"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            cost = time.perf_counter() - start
            print(f"{func.__name__} 耗时 {cost:.6f} 秒")
    return wrapper


@timer
def slow_sum(n):
    return sum(i * i for i in range(n))


print("结果:", slow_sum(1_000_000))


@timer
def will_fail():
    time.sleep(0.05)
    raise RuntimeError("boom")


try:
    will_fail()
except RuntimeError as e:
    print(f"捕获异常: {e}（注意耗时依然被打印了）")

slow_sum 耗时 0.026765 秒
结果: 333332833333500000
will_fail 耗时 0.053959 秒
捕获异常: boom（注意耗时依然被打印了）


### 8.2 重试 `retry`

要点：
- `exceptions` 限定**只对特定异常重试**，`KeyError` 这类逻辑错误不应重试。
- 最后一次失败必须 `raise`，不能静默吞掉异常。
- 生产环境建议使用**指数退避 + 抖动**，避免重试风暴。

In [21]:
import random


def retry(times=3, delay=1, exceptions=(Exception,), backoff=1.0):
    """失败重试装饰器，支持固定/指数退避。"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            wait = delay
            for i in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if i == times:
                        raise        # 最后一次仍然失败 → 抛出
                    print(f"{func.__name__} 第 {i} 次失败: {e}，{wait}s 后重试")
                    time.sleep(wait)
                    wait *= backoff  # backoff=1 固定间隔，>1 指数退避
        return wrapper
    return decorator


random.seed(11)   # 固定随机种子，保证输出可复现（前两次失败、第三次成功）

@retry(times=3, delay=0.1, exceptions=(ValueError,))
def unstable():
    if random.random() < 0.7:
        raise ValueError("随机失败")
    return "成功"


print("结果:", unstable())

unstable 第 1 次失败: 随机失败，0.1s 后重试


unstable 第 2 次失败: 随机失败，0.1s 后重试


结果: 成功


### 8.3 权限校验 `require_admin`

要点：`wrapper` 的第一个位置参数是 `user`（或 `self`），
因此**装饰器会约束被装饰函数的调用约定**——这类装饰器比通用装饰器耦合更强，
需要在文档里写清楚。

In [22]:
def require_admin(func):
    """要求第一个参数为 user 且 user['is_admin'] 为真。"""
    @wraps(func)
    def wrapper(user, *args, **kwargs):
        if not user.get("is_admin"):
            raise PermissionError(f"用户 {user.get('name')} 需要管理员权限")
        return func(user, *args, **kwargs)
    return wrapper


@require_admin
def delete_user(user, username):
    print(f"{user['name']} 删除了用户 {username}")
    return True


admin = {"name": "Alice", "is_admin": True}
normal = {"name": "Bob", "is_admin": False}

print(delete_user(admin, "tom"))

try:
    delete_user(normal, "tom")
except PermissionError as e:
    print("被拦截:", e)

Alice 删除了用户 tom
True
被拦截: 用户 Bob 需要管理员权限


### 8.4 命令注册器 `command`

要点：注册型装饰器**原样返回 func**（不包 wrapper），因为它的目的是「登记」而非「改写」。
这是 Web 框架路由（`@app.route`）、CLI 工具（`@click.command`）的共同基础。

In [23]:
commands = {}


def command(name):
    def decorator(func):
        commands[name] = func
        return func          # 原函数返回，调用行为完全不变
    return decorator


@command("add")
def add(a, b):
    """加法"""
    return a + b


@command("mul")
def mul(a, b):
    """乘法"""
    return a * b


print("已注册命令:", list(commands))
print("add:", commands["add"](2, 3))
print("mul:", commands["mul"](2, 3))

# 命令分发器
def dispatch(name, *args):
    if name not in commands:
        raise KeyError(f"未知命令: {name}")
    return commands[name](*args)

print("dispatch('add', 10, 20) =", dispatch("add", 10, 20))

已注册命令: ['add', 'mul']
add: 5
mul: 6
dispatch('add', 10, 20) = 30


### 8.5 缓存 `functools.lru_cache`

要点：
- 参数必须**可哈希**（list/dict/set 不行，需转成 tuple）。
- `maxsize=None` 表示无限缓存；默认 128，满了按 LRU 淘汰。
- `.cache_info()` 查看命中率，`.cache_clear()` 清空。
- Python 3.9+ 可用 `functools.cache`（等价于 `lru_cache(maxsize=None)`）。

In [24]:
from functools import lru_cache, cache


@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)


print("fib(100) =", fib(100))
print("缓存统计:", fib.cache_info())


# 参数必须可哈希 —— 反例
@lru_cache(maxsize=None)
def bad_sum(items):
    return sum(items)


try:
    bad_sum([1, 2, 3])
except TypeError as e:
    print("不可哈希参数报错:", e)


# 正确做法：转成 tuple
@lru_cache(maxsize=None)
def good_sum(items: tuple):
    return sum(items)

print("good_sum((1,2,3)) =", good_sum((1, 2, 3)))
print("第二次命中缓存:", good_sum((1, 2, 3)), good_sum.cache_info())

fib(100) = 354224848179261915075
缓存统计: CacheInfo(hits=98, misses=101, maxsize=None, currsize=101)
不可哈希参数报错: unhashable type: 'list'
good_sum((1,2,3)) = 6
第二次命中缓存: 6 CacheInfo(hits=1, misses=1, maxsize=None, currsize=1)


---

## 9. Python 内置常见装饰器

| 装饰器 | 作用 | 典型场景 |
| --- | --- | --- |
| `@staticmethod` | 不接收 `self`/`cls` 的实例无关方法 | 工具函数挂在类下 |
| `@classmethod` | 第一个参数是 `cls`（类本身） | 多态工厂方法 |
| `@property` | 把方法变属性访问 | 只读计算属性、校验 |
| `@dataclass` | 自动生成 `__init__`/`__repr__`/`__eq__` | DTO、配置对象 |
| `@lru_cache` | 记忆化缓存 | 递归、昂贵查询 |
| `@abstractmethod` | 抽象方法，强制子类实现 | 接口约束 |
| `@functools.wraps` | 编写装饰器时保留元信息 | 写自己的装饰器 |

它们本质上都是「接收函数/类，返回函数/类」——和我们自己写的没有区别。

In [25]:
class Math:
    """staticmethod 示例。"""

    @staticmethod
    def add(a, b):
        # 没有 self，也不需要实例化
        return a + b


print("Math.add(1, 2) =", Math.add(1, 2))
print("实例也能调用:", Math().add(3, 4))

Math.add(1, 2) = 3
实例也能调用: 7


In [26]:
from abc import ABC, abstractmethod


class User(ABC):
    def __init__(self, name):
        self.name = name

    @classmethod
    def from_dict(cls, data):
        """工厂方法：cls 是实际调用的类，子类继承后自动返回子类实例。"""
        return cls(data["name"])

    @abstractmethod
    def role(self):
        """子类必须实现。"""


class AdminUser(User):
    def role(self):
        return "admin"


print("AdminUser.from_dict 返回类型:", type(AdminUser.from_dict({"name": "root"})).__name__)
print("role():", AdminUser.from_dict({"name": "root"}).role())

try:
    User("x")     # 含 abstractmethod，不能被实例化
except TypeError as e:
    print("抽象类不可实例化:", e)

AdminUser.from_dict 返回类型: AdminUser
role(): admin
抽象类不可实例化: Can't instantiate abstract class User without an implementation for abstract method 'role'


In [27]:
class Circle:
    """property 示例：像属性一样访问方法，并支持 setter 校验。"""
    def __init__(self, radius):
        self._radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value <= 0:
            raise ValueError("半径必须为正数")
        self._radius = value

    @property
    def area(self):
        """只读属性（没有 setter）。"""
        return 3.14159 * self._radius ** 2


c = Circle(2)
print("area =", c.area)

c.radius = 5                      # 走 setter
print("改半径后 area =", c.area)

try:
    c.radius = -1
except ValueError as e:
    print("setter 拦截:", e)

try:
    c.area = 100                  # 只读属性
except AttributeError as e:
    print("只读属性不可赋值:", e)

area = 12.56636
改半径后 area = 78.53975
setter 拦截: 半径必须为正数
只读属性不可赋值: property 'area' of 'Circle' object has no setter


In [28]:
from dataclasses import dataclass, field


@dataclass
class UserDTO:
    name: str
    age: int = 0
    tags: list = field(default_factory=list)   # 可变默认值必须用 default_factory


u1 = UserDTO("Alice")
u2 = UserDTO("Alice")
print(u1)                 # 自动生成 __repr__
print("自动 __eq__:", u1 == u2)

u1.tags.append("vip")
u2.tags.append("new")
print("各自独立的 list:", u1.tags, u2.tags)


@dataclass(frozen=True)
class Config:
    model: str
    temperature: float = 0.7


cfg = Config("gpt-4o")
print("frozen dataclass:", cfg)
try:
    cfg.model = "qwen"
except Exception as e:
    print("不可变:", type(e).__name__, e)

UserDTO(name='Alice', age=0, tags=[])
自动 __eq__: True
各自独立的 list: ['vip'] ['new']
frozen dataclass: Config(model='gpt-4o', temperature=0.7)
不可变: FrozenInstanceError cannot assign to field 'model'


---

## 10. 异步函数装饰器

三条铁律：

1. 被装饰函数是 `async def` → `wrapper` **也必须是** `async def`，否则返回的是协程对象而非结果。
2. 包装函数内部必须 `await func(...)`，否则协程**根本不会执行**。
3. 装饰器返回的可调用对象仍是协程函数，`await` 语义不变。

> 在 Jupyter 中可以直接在单元格顶层 `await`；
> 如果放到 `.py` 脚本里，请改用 `asyncio.run(...)`。

In [29]:
import asyncio


def async_timer(func):
    """异步版计时器。"""
    @wraps(func)
    async def wrapper(*args, **kwargs):
        loop = asyncio.get_running_loop()
        start = loop.time()
        try:
            return await func(*args, **kwargs)
        finally:
            cost = loop.time() - start
            print(f"{func.__name__} 耗时 {cost:.6f} 秒")
    return wrapper


@async_timer
async def task(name, sec):
    await asyncio.sleep(sec)
    return f"{name} 完成"


print(await task("任务A", 0.3))

# 并发：装饰器不改变并发语义
results = await asyncio.gather(task("B", 0.2), task("C", 0.2))
print("并发结果:", results)

task 耗时 0.300973 秒
任务A 完成


task 耗时 0.201652 秒
task 耗时 0.201844 秒
并发结果: ['B 完成', 'C 完成']


In [30]:
# ---------- 反例：wrapper 忘记 async/await ----------
def wrong_async_deco(func):
    @wraps(func)
    def wrapper(*args, **kwargs):     # 普通函数，没有 await
        print("wrong before")
        return func(*args, **kwargs)  # 返回的是协程对象，函数体并未执行
    return wrapper


@wrong_async_deco
async def never_runs():
    print("  <<< 这行永远不会打印 >>>")
    await asyncio.sleep(0.1)
    return "done"


r = never_runs()
print("返回值类型:", type(r), "->", r)
print("是否协程:", asyncio.iscoroutine(r))
await r        # 只有手动 await 才会真正执行

wrong before
返回值类型: <class 'coroutine'> -> <coroutine object never_runs at 0x1074d1f00>
是否协程: True
  <<< 这行永远不会打印 >>>


'done'

In [31]:
# 异步重试：与同步版结构一致，只是 await
def async_retry(times=3, delay=0.1, exceptions=(Exception,)):
    def decorator(func):
        @wraps(func)
        async def wrapper(*args, **kwargs):
            for i in range(1, times + 1):
                try:
                    return await func(*args, **kwargs)
                except exceptions as e:
                    if i == times:
                        raise
                    print(f"  第 {i} 次失败: {e}，{delay}s 后重试")
                    await asyncio.sleep(delay)   # 注意用 asyncio.sleep，不能用 time.sleep
        return wrapper
    return decorator


state = {"n": 0}

@async_retry(times=3, delay=0.05)
async def flaky_call():
    state["n"] += 1
    if state["n"] < 3:
        raise ConnectionError("连接超时")
    return "OK"


print("结果:", await flaky_call())

  第 1 次失败: 连接超时，0.05s 后重试
  第 2 次失败: 连接超时，0.05s 后重试


结果: OK


---

## 11. 六个高频陷阱

| # | 陷阱 | 症状 | 解法 |
| --- | --- | --- | --- |
| 1 | 忘记 `return result` | 调用方拿到 `None` | `return func(...)` |
| 2 | 忘记 `@wraps` | `__name__` 变 `wrapper`，签名丢失 | 加 `@wraps(func)` |
| 3 | 参数写死 | 换函数就报错 | `*args, **kwargs` |
| 4 | 误判执行时机 | 以为装饰器在调用时执行 | 装饰在**定义时**执行 |
| 5 | 顺序搞反 | 日志/鉴权层级错乱 | `f = deco1(deco2(f))` |
| 6 | 装饰方法时忽略 `self` | 少收一个参数 | wrapper 用 `*args` 透传 |

下面逐个复现。

In [32]:
# 陷阱 1：忘记返回结果（已在第 2 节演示，这里再强调一次）
def no_return(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)
    return wrapper


@no_return
def get_user():
    return {"id": 1, "name": "Alice"}


print("陷阱1 →", get_user())


# 陷阱 2：忘记 wraps
def no_wraps(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@no_wraps
def documented(x: int) -> int:
    """有文档的函数。"""
    return x


print("陷阱2 → __name__:", documented.__name__, "| __doc__:", documented.__doc__)
print("        signature:", inspect.signature(documented))

陷阱1 → None
陷阱2 → __name__: wrapper | __doc__: None
        signature: (*args, **kwargs)


In [33]:
# 陷阱 3：wrapper 参数写死，导致装饰器不通用
def rigid(func):
    def wrapper(a, b):          # 只能接两个位置参数
        return func(a, b)
    return wrapper


@rigid
def three_args(a, b, c):
    return a + b + c


try:
    three_args(1, 2, 3)
except TypeError as e:
    print("陷阱3 →", e)


# 正确写法
def flexible(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@flexible
def three_args_ok(a, b, c):
    return a + b + c

print("正确写法 →", three_args_ok(1, 2, 3))

陷阱3 → rigid.<locals>.wrapper() takes 2 positional arguments but 3 were given
正确写法 → 6


In [34]:
# 陷阱 4：装饰器在「定义时」就执行，不是调用时
print("=== 模块加载阶段 ===")

def eager(func):
    print(f"  [装饰器执行] 正在装饰 {func.__name__}")   # 这一行在定义时立刻打印
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"  [wrapper 执行] 每次调用都会打印")
        return func(*args, **kwargs)
    return wrapper


@eager
def business():
    return "业务逻辑"


print("=== 调用阶段 ===")
business()
business()
print("结论：'[装饰器执行]' 只出现 1 次，'[wrapper 执行]' 出现 2 次")

=== 模块加载阶段 ===
  [装饰器执行] 正在装饰 business
=== 调用阶段 ===
  [wrapper 执行] 每次调用都会打印
  [wrapper 执行] 每次调用都会打印
结论：'[装饰器执行]' 只出现 1 次，'[wrapper 执行]' 出现 2 次


In [35]:
# 陷阱 6：装饰实例方法时，self 会作为 args[0] 进入 wrapper
def method_logger(func):
    @wraps(func)
    def wrapper(*args, **kwargs):     # args[0] 就是 self
        self_obj = args[0]
        print(f"  调用 {self_obj.__class__.__name__}.{func.__name__}")
        return func(*args, **kwargs)
    return wrapper


class OrderService:
    @method_logger
    def create(self, order_id):
        return f"order {order_id} created"


print("正确写法（*args 透传 self）:", OrderService().create("A-001"))


# 反例：wrapper 参数写死，漏掉了 self
def buggy_logger(func):
    @wraps(func)
    def wrapper(order_id):            # 少收一个参数
        return func(order_id)
    return wrapper


class BuggyService:
    @buggy_logger
    def create(self, order_id):
        return f"order {order_id} created"


try:
    BuggyService().create("A-001")
except TypeError as e:
    print("陷阱6 →", e)

  调用 OrderService.create
正确写法（*args 透传 self）: order A-001 created
陷阱6 → BuggyService.create() takes 1 positional argument but 2 were given


In [36]:
# 陷阱 5：与 @property / @staticmethod / @classmethod 叠加时的顺序
# 规则：装饰器在 @property 之上时，它拿到的是 property **描述符对象**，
#       而不是原始函数 —— 行为会悄悄改变，且不报错，最难排查。

class BadOrder:
    @log_calls          # ❌ log_calls 在外层：包装的是 property 对象
    @property
    def value(self):
        return 42


class GoodOrder:
    @property           # ✅ property 在外层：log_calls 包装的是真实函数
    @log_calls
    def value(self):
        return 42


bad, good = BadOrder(), GoodOrder()
print("错误顺序 p.value  =", bad.value)
print("                     ↑ 属性取值变成了 bound method，结果错误")

try:
    bad.value()          # 再调用一次才会暴露真正的错误
except TypeError as e:
    print("错误顺序 p.value() →", e)

print("正确顺序 p.value  =", good.value)

错误顺序 p.value  = <bound method log_calls.<locals>.wrapper of <__main__.BadOrder object at 0x1077a6ba0>>
                     ↑ 属性取值变成了 bound method，结果错误
[LOG] 调用 value(args=(<__main__.BadOrder object at 0x1077a6ba0>,), kwargs={})
错误顺序 p.value() → 'property' object is not callable
[LOG] 调用 value(args=(<__main__.GoodOrder object at 0x1077a6cf0>,), kwargs={})
[LOG] value 返回 42
正确顺序 p.value  = 42


---

## 12. 综合实战：LLMOps 场景中的装饰器组合

需求：给一个「调用大模型」的函数叠加以下能力，且**顺序不能错**：

```
@register_tool("translate")     ← 第 1 层：登记到工具表（返回原函数，透明）
@lru_cache(maxsize=128)         ← 第 2 层：缓存，命中即短路下面全部逻辑
@log_call                       ← 第 3 层：记录入参出参
@retry(times=3, ...)            ← 第 4 层：网络抖动重试
@timer                          ← 第 5 层（最内）：统计单次调用耗时
def llm_translate(text, target): ...
```

**顺序设计理由（调用链自外向内：`register → cache → log → retry → timer → func`）：**

- `register_tool` 放最外：它只是登记、原样返回函数，不改变任何调用行为，放最上最清晰。
- `lru_cache` 放在**外侧**才能「短路」：命中缓存时直接返回，**日志 / 重试 / 计时都不会执行**。
  如果放到最内层，缓存虽生效，但日志与计时仍会被触发（可自行调换顺序验证）。
- `retry` 在 `timer` 外层 → `timer` 统计的是**单次尝试**的耗时；
  若想统计**含重试的总耗时**，应把 `timer` 放到 `retry` 外面（见下一单元对比）。
- `lru_cache` 要求参数可哈希；它返回的是 C 实现对象，
  因此放在**最外层**时，上层装饰器（这里是 register）不要依赖 `__wrapped__` 链。

In [37]:
# ---------- 1. 工具注册表 ----------
TOOLS = {}


def register_tool(name):
    def decorator(func):
        TOOLS[name] = func
        return func
    return decorator


# ---------- 2. 日志装饰器 ----------
def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG ] → {func.__name__}{args} {kwargs}")
        result = func(*args, **kwargs)
        print(f"[LOG ] ← {result!r}")
        return result
    return wrapper


# ---------- 3. 模拟 LLM 客户端 ----------
class MockLLM:
    def __init__(self, fail_rate=0.5):
        self.fail_rate = fail_rate
        self.n = 0

    def invoke(self, prompt):
        self.n += 1
        r = random.Random(self.n).random()
        if r < self.fail_rate:
            raise ConnectionError(f"上游超时 (call #{self.n})")
        time.sleep(0.02)
        return f"<<{prompt} 的译文>>"


llm = MockLLM(fail_rate=0.4)


# ---------- 4. 组装 ----------
@register_tool("translate")
@lru_cache(maxsize=128)
@log_call
@retry(times=3, delay=0.02, exceptions=(ConnectionError,))
@timer
def llm_translate(text, target="zh"):
    """调用大模型翻译。"""
    return llm.invoke(f"把 {text!r} 翻译成 {target}")


print("=== 第一次调用（可能触发重试） ===")
print("结果:", llm_translate("hello", "zh"))

print("\n=== 第二次调用（相同参数 → 命中缓存，日志/重试/计时全部跳过） ===")
print("结果:", llm_translate("hello", "zh"))

print("\n=== 第三次调用（不同参数 → 重新走完整链路） ===")
print("结果:", llm_translate("world", "en"))

print("\n工具表:", list(TOOLS))
print("工具可直接调用:", TOOLS["translate"]("cached", "zh"))
print("缓存统计（注意：命中时并未触发日志与计时）:", llm_translate.cache_info())

=== 第一次调用（可能触发重试） ===
[LOG ] → llm_translate('hello', 'zh') {}
llm_translate 耗时 0.000020 秒
llm_translate 第 1 次失败: 上游超时 (call #1)，0.02s 后重试


llm_translate 耗时 0.023305 秒
[LOG ] ← "<<把 'hello' 翻译成 zh 的译文>>"
结果: <<把 'hello' 翻译成 zh 的译文>>

=== 第二次调用（相同参数 → 命中缓存，日志/重试/计时全部跳过） ===
结果: <<把 'hello' 翻译成 zh 的译文>>

=== 第三次调用（不同参数 → 重新走完整链路） ===
[LOG ] → llm_translate('world', 'en') {}
llm_translate 耗时 0.000033 秒
llm_translate 第 1 次失败: 上游超时 (call #3)，0.02s 后重试
llm_translate 耗时 0.000057 秒
llm_translate 第 2 次失败: 上游超时 (call #4)，0.02s 后重试
llm_translate 耗时 0.021201 秒
[LOG ] ← "<<把 'world' 翻译成 en 的译文>>"
结果: <<把 'world' 翻译成 en 的译文>>

工具表: ['translate']
[LOG ] → llm_translate('cached', 'zh') {}
llm_translate 耗时 0.025060 秒
[LOG ] ← "<<把 'cached' 翻译成 zh 的译文>>"
工具可直接调用: <<把 'cached' 翻译成 zh 的译文>>
缓存统计（注意：命中时并未触发日志与计时）: CacheInfo(hits=1, misses=3, maxsize=128, currsize=3)


In [38]:
# 装饰器顺序对结果的影响：对比 retry 与 timer 的相对位置
print("=== A: retry 在 timer 外层 → 每次尝试各计一次耗时 ===")

@retry(times=2, delay=0.05, exceptions=(ConnectionError,))
@timer
def a_call():
    raise ConnectionError("fail")

try:
    a_call()
except ConnectionError as e:
    print("最终失败:", e)

print("\n=== B: timer 在 retry 外层 → 一次统计含重试的总耗时 ===")

@timer
@retry(times=2, delay=0.05, exceptions=(ConnectionError,))
def b_call():
    raise ConnectionError("fail")

try:
    b_call()
except ConnectionError as e:
    print("最终失败:", e)

=== A: retry 在 timer 外层 → 每次尝试各计一次耗时 ===
a_call 耗时 0.000002 秒
a_call 第 1 次失败: fail，0.05s 后重试


a_call 耗时 0.000004 秒
最终失败: fail

=== B: timer 在 retry 外层 → 一次统计含重试的总耗时 ===
b_call 第 1 次失败: fail，0.05s 后重试


b_call 耗时 0.055042 秒
最终失败: fail


---

## 13. 速查表与练习

### 通用模板

```python
from functools import wraps

# 无参数装饰器
def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # before
        result = func(*args, **kwargs)
        # after
        return result
    return wrapper


# 带参数装饰器（多一层）
def decorator_factory(param):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        return wrapper
    return decorator


# 类装饰器
class Decorator:
    def __init__(self, func):
        self.func = func
        update_wrapper(self, func)
    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)


# 异步装饰器
def async_decorator(func):
    @wraps(func)
    async def wrapper(*args, **kwargs):
        return await func(*args, **kwargs)
    return wrapper
```

### 记忆口诀

1. **装饰器 = 可调用对象，接收函数，返回函数。**
2. **要传参，多一层。**
3. **wrapper 用 `*args/**kwargs`，并且一定要 `return`。**
4. **写装饰器必加 `@wraps`。**
5. **多个装饰器：`f = deco1(deco2(f))`，近的先包装、远的执行时先出场。**
6. **装饰器在定义时执行一次，wrapper 在每次调用时执行。**

### 自测练习

1. 写一个 `@memoize` 装饰器，手动实现缓存（不用 `lru_cache`），
   要求支持关键字参数，并暴露 `cache_clear()` 方法（提示：用类实现）。
2. 写一个 `@validate` 装饰器，根据函数类型注解校验入参类型，
   不匹配时抛 `TypeError`（提示：`inspect.signature` + `typing.get_type_hints`）。
3. 写一个 `@singledispatch`-风格的路由装饰器，让同名函数根据第一个参数类型分发。
4. 把第 12 节的链路改成：`@timer` 在最外层，`@lru_cache` 在最内层，
   观察缓存命中时的输出差异，并解释原因。
5. 写一个 `@deprecated(reason)` 装饰器，被调用时发出 `DeprecationWarning`。

In [39]:
# 练习 1 参考实现：用类实现可清空的 memoize
class Memoize:
    """支持关键字参数的记忆化装饰器，提供 cache_clear()。"""
    def __init__(self, func):
        self.func = func
        self.cache = {}
        update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in self.cache:
            self.cache[key] = self.func(*args, **kwargs)
        return self.cache[key]

    def cache_clear(self):
        self.cache.clear()


@Memoize
def expensive(x, scale=1):
    print(f"  （真正计算 expensive({x}, scale={scale})）")
    return x * x * scale


print(expensive(4, scale=2))
print(expensive(4, scale=2))      # 命中缓存，不再打印
print(expensive(5, scale=2))
print("缓存条目数:", len(expensive.cache))
expensive.cache_clear()
print("清空后缓存条目数:", len(expensive.cache))

  （真正计算 expensive(4, scale=2)）
32
32
  （真正计算 expensive(5, scale=2)）
50
缓存条目数: 2
清空后缓存条目数: 0


In [40]:
# 练习 5 参考实现：@deprecated
import warnings


def deprecated(reason=""):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            warnings.warn(
                f"{func.__name__} 已废弃{'：' + reason if reason else ''}",
                DeprecationWarning,
                stacklevel=2,
            )
            return func(*args, **kwargs)
        return wrapper
    return decorator


@deprecated("请改用 new_api()")
def old_api():
    return "old result"


with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    print(old_api())
    print("捕获警告:", str(w[0].message))

old result
捕获警告: old_api 已废弃：请改用 new_api()


---

## 小结

装饰器的价值不在语法，而在于它提供了一种**不侵入业务代码即可横切增强**的能力。
掌握它之后，框架里的这些机制都会变得透明：

- 路由注册：Flask `@app.route`、FastAPI `@app.get`
- 依赖注入：`@inject`
- 权限与鉴权：`@login_required`
- 缓存：`@lru_cache`、`@cache`
- 重试与容错：`@retry`、LangChain 的 `with_retry()`
- 事务与会话：SQLAlchemy `@event.listens_for`、Django `@transaction.atomic`
- 任务编排：Celery `@app.task`、Airflow `@dag` / `@task`

**下一步建议**：挑一个你正在用的框架（Flask / FastAPI / LangChain），
阅读其中一个装饰器（如 `@app.route`）的源码，验证它与本文模板的一致性。